## Meteorological computations

In this notebook you will see:
- how to compute the potential temperature from GRIB data
- how to compute the wind speed from GRIB data

### Components of earthkit

This tutorial uses the following earthkit components - click any logo to open the package documentation:

<div align="center">
  <br>
  <a href="https://earthkit-data.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-data-light.svg" alt="earthkit-data" width="200">
  </a>
  <a href="https://earthkit-meteo.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-meteo-light.svg" alt="earthkit-meteo" width="200">
  </a>
  <a href="https://earthkit-plots.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-plots-light.svg" alt="earthkit-plots" width="200">
  </a>
</div>

### 1. Getting the data

Get the input data containing temperature, specific humidity and wind analysis on pressure levels.

In [ ]:
import earthkit.data as ekd

ds = ekd.from_source("sample", "tquv_pl_2x2.grib").to_fieldlist()
ds.head(6)

### 2. Computing potential temperature

This example shows how to compute the potential temperature with [earthkit.meteo.thermo.potential_temperature](https://earthkit-meteo.readthedocs.io/en/latest/autoapi/earthkit/meteo/thermo/array/potential_temperature.html#earthkit.meteo.thermo.array.potential_temperature).

In [ ]:
from earthkit.meteo import thermo

# select temperature fields
t = ds.sel({"parameter.variable": "t"})

res = []
for f in t:
    # do the computation
    pres_pa = f.get("vertical.level") * 100
    # TODO: # why metadata.shortName still 't'  even after this? Looks like cache issue
    # TODO: do we need to also set paramId somehow?
    # TODO: actually add fieldlist support that sets all this correctly in ekm
    result = thermo.potential_temperature(f, pres_pa).set({"parameter.variable": "pt"})
    res.append(result)

# # create the resulting fieldlist    
ds_pt = ekd.FieldList.from_fields(res)

Check the results.

In [ ]:
ds_pt.ls()

The resulting fieldlist was created in memory. You can save it into a file with `to_target()`.

In [ ]:
# TODO: check why need to write and then reread in order to get correct metadata for shortName etc.
# also needed for the plots to have correct labels
ds_pt.to_target("file", "_pt_res.grib")

# read back saved data and check first 2 fields
ds_pt = ekd.from_source("file", "_pt_res.grib").to_fieldlist()
ds_pt.head(2)

The next cell plots the input temperature and the computed potential temperature fields on the same level.

In [ ]:
import earthkit.plots as ekp

figure = ekp.Figure(rows=1, columns=2)

level = 850
t_style = ekp.styles.Style(
    units="K", levels=list(range(220,340,10))
)

subplot = figure.add_map(0, 0)
subplot.contourf(t.sel({"vertical.level": level}), style=t_style)
subplot.title("{shortName} {level} hPa")
subplot.legend()

subplot = figure.add_map(0, 1)
subplot.contourf(ds_pt.sel({"vertical.level": level}), style=t_style)
subplot.title("{shortName} {level} hPa")
subplot.legend()

figure.coastlines()
figure.gridlines()

figure.show()

### 3. Computing the wind speed

This example computes the wind speed with  [earthkit.meteo.wind.speed](https://earthkit-meteo.readthedocs.io/en/latest/autoapi/earthkit/meteo/wind/array/speed.html#earthkit.meteo.wind.array.speed).

In [ ]:
ds = ekd.from_source("sample", "tquv_pl_2x2.grib").to_fieldlist()

In [ ]:
from earthkit.meteo import wind

# select the u and v fields. We assume here they
# are valid for the same set of levels
u = ds.sel({"parameter.variable": "u"})
v = ds.sel({"parameter.variable": "v"})

res = []


# loop and perform the computation per field, i.e per level
for f_u, f_v in zip(u, v):
    # do the computation
    result = wind.speed(f_u, f_v).set({"parameter.variable": "ws"})
    
    res.append(result)

# create the resulting fieldlist
ds_ws = ekd.FieldList.from_fields(res)

Check the results.

In [ ]:
ds_ws.ls()

Write out the results and read back in.

In [ ]:
ds_ws.to_target("file", "_ws_res.grib")

# read back saved data and check first 2 fields
ds_ws = ekd.from_source("file", "_ws_res.grib").to_fieldlist()
ds_ws.head(2)

The next cell plots the wind arrows and the computed speed over a subarea for a given level.

In [ ]:

level = 850

chart = ekp.Map(domain="Europe", size=(6,6))
chart.contourf(ds_ws.sel({"vertical.level": level}), units="m s-1", colors="Greens", 
               levels=list(range(4,22,2)), alpha=1)
chart.quiver(u=u.sel({"vertical.level": level}), v=v.sel({"vertical.level": level}))
chart.coastlines()
chart.gridlines()
chart.legend()
chart.title(("{time:%Y-%m-%d %H} UTC (+{lead_time}h) {level} hPa"))

chart.show()